---
toc: false
title: Why Causality — the Prediction-Decision Gap
date: "10/30/2026"
description: Correlation is not causation, except when it is.
categories: causal_inference
image: /enso-thumbnail.jpg
image-alt: The enso circle
draft: true
---


## Why causal inference 

### The Purpose of Data Science

Every once in a while my mom asks me if I can re-explain to her what data scientists do. It's a great question really—we in the field should ask ourselves more often. In a single sentence I'd say: the purpose of data science is to create value from data.  And what's valuable to the organizations we serve?  Well an organization has goals, and in order to achieve those goals it needs to make decisions about which actions to take. At its most effective, data science is about decision intelligence—informing a decision making process to help achieve desired outcomes. These decisions come in many forms, from executive deliberation on strategic bets to streaming automations occurring millions of times per day.

Examples include:
* What price to charge for this ride share trip?
* Which product to recommend to this user?
* Should a school system reduce its class sizes?
* Where to deploy mobile health centers in an underserved rural area?

Behind each of these decisions is a set of what if questions of the form: What would happen if I did this? What would happen if I did that? To inform the decision we can predict answers to these what if questions. 

Often these answers are already valuable, but we can go one step further and recommend the action corresponding to the most favorable outcome. Thus our business problems can often be formulated as optimization problems of the form

$$ \underset{x}{\operatorname{argmax}} f(x) $$

where $x$ indexes the possible actions we can take and $f$ is a "what if" function that predicts the benefit of an action. The above expression says: "find the action that maximizes the predicted benefit."

So that's the setup; the key for optimal decision making is having a "what if" function that reliably maps actions to outcomes.

For data scientists of my generation who spent the early part of their careers getting good at machine learning, this probably  sounds an awful lot like a prediction problem.
We could follow the typical ML playbook , training a model that minimizes out of sample prediction error.
That means it would be the best predictor of  outcomes from actions  and an ideal objective function to use in our optimization task right?

Maybe; maybe not.

Unfortunately there are a few ways this can go right and  many ways this can go wrong. 
Using well-fitting statistical models or accuracy-maxing ML models for decision making can fail, not just slightly, but catastrophically. 
Many of these failure modes arise from errors in reasoning about what causes what and how, i.e. causality.
In this post we'll look at two types of prediction-decision failure modes that can occurr because of mishandling causality.
The first is about the importance of the causal mechanisms behind the correlations we exploit for prediction, and the second is about how these mechanisms can shift under our feet when taking a model from one environment to another.
The key idea is that using statistical or ML models for decision making requires careful causal reasoning, and lucky for you, this is the first post in a forthcoming series on Causal Inference.

So let's kick this series off by seeing some of the scary things that can happen when we naively use predictive models for decision making.

## Prediction-Decision Failure Modes

### Correlation does not imply causation — except when it does.

We've all had it drilled into our heads — 
*[correlation does not imply causation](https://en.wikipedia.org/wiki/Correlation_does_not_imply_causation)* —
meaning it isn't possible to  deduce a cause-and-effect relationship between two events  solely on the basis of an observed correlation or other statistical association between them.
This does not mean there isn't a cause effect relationship between the correlated variables; it just means we can't tell what kind of relationship it is.
To put a finer point on this, consider [Reichenbach's Common Cause Principle](https://plato.stanford.edu/entries/physics-Rpcc/), 
which states that if two events are correlated, there must either be a causal relationship between them or a third event that causes both.
That means that a true correlation between two variables tells us that one causes the other, the other causes the one, or another variable causes both. 
So in a sense, correlation *does* imply causation, but alone it cannot tell us *which* causal relationship exists between the variables.

Let that sink in, this means that no matter how much observed data we collect, it's not possible from the data alone to determine which variables are causes and which are effects.
Furthermore, since predictive models simply leverage correlations and associations among variables from a sample of training data, a predictive model cannot tell us what causes what.
This isn't a problem for prediction, but in decision making, what causes what is absolutely critical.
Operating with the wrong causal mechanism in mind can lead us to make misguided, suboptimal, or even terrible decisions.


> Maybe "correlation does not imply causation"  reminds you of the
[spurious correlations](https://www.tylervigen.com/spurious-correlations)
between unrelated quantities like the  popularity  of the surprised Pikachu meme and the number of middle school teachers in Puerto Rico.
These variables are not truly correlated at a population or process level; they are cherry picked by dredging sample correlations computed on small samples of unrelated variables, keeping those that happen to show high correlation by chance.
In the context of this post, we're setting aside spurious correlations and talking about variables which are actually correlated in a population sense, not just in a particular finite sample.


### Wrong Causal Mechanism


Suppose we're looking at some app useage data, and we want to figure out whether notifications can help to increase user engagement. 

* X: user enables push notifications during their first two weeks.
* Y: engagement, measured by the number of sessions in the first two weeks.

The data shows  a strong positive correlation between notifications and sessions. 
We build a predictive model that predicts higher engagement when users have notifications enabled.
Our goal is to increase engagement, and we need to make a decision about how to do that — specifically, *should we prompt every new user on day 1 to turn notifications on*?
Well the model predicts that more users with notifications enabled gives more engagement, so naively using the predictive model, we'd decide to go ahead and prompt every user to enable on day 1.
There are many different causal mechanisms that could produce the observed correlation between notifications and sessions; let's look at a few of them and what it would mean for our prediction-based decision to prompt users on day 1.

#### 1. Notifications Cause Engagement

Suppose notifications remind people to come back, so each enabled user opens the app more often.

Structure: Notifications → Sessions

If we prompt users on day 1, sessions go up — we made the right decision!

### 2. Reverse Causation: Engagement Causes Sessions

Suppose the default OS notification permission prompt automatically fires  after a user’s 5th session. 
Engaged users with many sessions see the prompt and many of them accept, while users who only open the app a few times never see it and thus few of them enable notifications.
Suppose further the notifications themselves have no effect on sessions.

Structure: Sessions → Sees prompt → Notifications on

Under this mechanism, sessions cause the prompt to fire, which causes notifications.


If we prompt users to enable notifications on day 1, sessions won't actually increase.


### 3. Common cause: invites cause notifications and engagement

Suppose many users discover the app through an invite link sent by a  friend. 
They enable notifications to get updated on what their friends are doing, and they also use the app more than non-invite users because of that social pull. 
Again, notifications don't actually increase engagement.

Structure: Notifications ← Invite → Sessions

If we prompt users to enable notifications on day 1, more non-invited users enable notifications, but this doesn't actually affect engagement.

### 4. Common cause with annoying notifications

Suppose we have the same common cause mechanism as before but now the notifications are actually slightly annoying to users, causing a small reduction in engagement.

Structure: Notifications ← Invite → Sessions, Notifications --> Sessions

Note that because the tendency of invite to increase both notifications and engagement is stronger than the tendency of notifications to reduce sessions, we'll still see that positive correlation between notifications and sessions in the data and predictive model.
If we prompt users to enable notifications on day 1, we actually end up decreasing engagement.

Under all four of these causal mechanisms, the data and predictive model would show a positive correlation between notifications and engagement, but choosing to prompt users to enable notifications on day 1 would only help under the first one.
Under the second two we'd produce no effect and under the last one we'd actually achieve the exact opposite of what we intended — these are the bitter pitfalls of the prediction-decision gap.

## Failure to Generalize Across Environments

## Looking Ahead